Build the cube!
Need to take point locations from:
"C:\NCA_DATA\Vegetation Data\Master_PointFeatures_26911.csv"

and intersect with the following datasets

vegetation:
"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv"

LTDL
"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_written"
with lookup tables: "C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_lookup_tables"

Polaris
"C:\NCA_DATA\Ancillary_Data\POLARIS_26911_10m_mosaic.tif"

Topo
"C:\NCA_DATA\Ancillary_Data\USGS3DEP_topo_26911_10m.tif"

MTBS
"D:\My Drive\S2_Harmonics_RCEW\MTBS_BurnContext_Schema.csv"
"D:\My Drive\S2_Harmonics_RCEW\MTBS_BurnContext_Multiband_10m.tif"
with categorical labels:
MTBS_CATEGORICAL_LABELS = {
    "mtbs_ever_burned": {
        0: "not_burned",
        1: "burned",
    },
    "mtbs_most_recent_severity": {
        0: "background",
        1: "unburned_to_low",
        2: "low",
        3: "moderate",
        4: "high",
        5: "increased_greenness",
        6: "non_mapping_area",
        255: "nodata",  # your pipeline fill value, not MTBS native class
    },
}

MTBS_CONTINUOUS_OR_COUNT_VARS = {
    "mtbs_most_recent_burn_year": {
        "nodata": -32768,
        "units": "year",
    },
    "mtbs_time_since_most_recent_burn": {
        "nodata": -32768,
        "units": "years",
    },
    "mtbs_burn_count": {
        "nodata": None,  # 0 is valid
        "units": "count",
    },
}

In [1]:
# ============================================================
# BUILD 2025 POINT DATA CUBE
# ============================================================

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import rasterio


# ============================================================
# PATHS
# ============================================================

points_csv = Path(r"C:\NCA_DATA\Vegetation Data\Master_PointFeatures_26911.csv")
veg_csv    = Path(r"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv")

ltdl_dir   = Path(r"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_written")
ltdl_lookup_dir = Path(r"C:\NCA_DATA\Ancillary_Data\LTDL\LTDL_lookup_tables")

spectral_tif = Path(r"C:\NCA_DATA\S2_Harmonics_vrt\S2SR_Harmonics_2025_4th_amp_phase_masked.tif")

polaris_tif = Path(r"C:\NCA_DATA\POLARIS_vrt\POLARIS_26911_10m_mosaic_exact.tif")
topo_tif    = Path(r"C:\NCA_DATA\Ancillary_Data\USGS3DEP_topo_26911_10m.tif")

mtbs_schema_csv = Path(r"C:\NCA_DATA\Ancillary_Data\MTBS\MTBS_BurnContext_Schema.csv")
mtbs_tif        = Path(r"C:\NCA_DATA\Ancillary_Data\MTBS\MTBS_BurnContext_Multiband_10m.tif")

out_dir = Path(r"C:\NCA_DATA\Point_Cubes")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "NCA_2025_point_cube_noNLDAS.csv"


# ============================================================
# USER SETTINGS
# ============================================================

YEAR_KEEP = 2025
X_FIELD = "X"
Y_FIELD = "Y"
YEAR_FIELD_CANDIDATES = ["Year", "year", "YEAR", "SampleYear", "sample_year"]

POLARIS_BAND_NAMES = [
    "sand_0_5", "sand_5_15", "sand_15_30", "sand_30_60", "sand_60_100", "sand_100_200",
    "silt_0_5", "silt_5_15", "silt_15_30", "silt_30_60", "silt_60_100", "silt_100_200",
    "clay_0_5", "clay_5_15", "clay_15_30", "clay_30_60", "clay_60_100", "clay_100_200",
    "bd_0_5", "bd_5_15", "bd_15_30", "bd_30_60", "bd_60_100", "bd_100_200",
    "theta_s_0_5", "theta_s_5_15", "theta_s_15_30", "theta_s_30_60", "theta_s_60_100", "theta_s_100_200",
    "ph_0_5", "ph_5_15", "ph_15_30", "ph_30_60", "ph_60_100", "ph_100_200",
    "om_log10_0_5", "om_log10_5_15", "om_log10_15_30", "om_log10_30_60", "om_log10_60_100", "om_log10_100_200",
]

MTBS_CATEGORICAL_LABELS = {
    "mtbs_ever_burned": {
        0: "not_burned",
        1: "burned",
    },
    "mtbs_most_recent_severity": {
        0: "background",
        1: "unburned_to_low",
        2: "low",
        3: "moderate",
        4: "high",
        5: "increased_greenness",
        6: "non_mapping_area",
        255: "nodata",
    },
}

MTBS_CONTINUOUS_OR_COUNT_VARS = {
    "mtbs_most_recent_burn_year": {
        "nodata": -32768,
        "units": "year",
    },
    "mtbs_time_since_most_recent_burn": {
        "nodata": -32768,
        "units": "years",
    },
    "mtbs_burn_count": {
        "nodata": None,
        "units": "count",
    },
}


# ============================================================
# HELPERS
# ============================================================

def find_field(df, candidates, required=True):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    if required:
        raise ValueError(f"None of these fields were found: {candidates}")
    return None


def clean_name(name):
    name = str(name).strip()
    name = re.sub(r"[^\w]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


def get_band_names(src, fallback_prefix):
    names = []
    for i in range(1, src.count + 1):
        desc = src.descriptions[i - 1]
        if desc is None or str(desc).strip() == "":
            names.append(f"{fallback_prefix}_band_{i:02d}")
        else:
            names.append(clean_name(desc))
    return names


def sample_multiband_raster(tif_path, points_df, x_field, y_field, prefix=None, band_names=None):
    tif_path = Path(tif_path)

    if prefix is None:
        prefix = clean_name(tif_path.stem)

    coords = list(zip(points_df[x_field].astype(float), points_df[y_field].astype(float)))

    with rasterio.open(tif_path) as src:
        if band_names is None:
            band_names = get_band_names(src, prefix)
        else:
            band_names = [clean_name(b) for b in band_names]

        if len(band_names) != src.count:
            raise ValueError(
                f"Band-name mismatch for {tif_path.name}: "
                f"{len(band_names)} names for {src.count} bands."
            )

        arr = np.array(list(src.sample(coords)))

        nodata = src.nodata
        out = pd.DataFrame(arr, columns=[f"{prefix}_{b}" for b in band_names])

        if nodata is not None:
            out = out.replace(nodata, np.nan)

    return out


def read_lookup_table(path):
    path = Path(path)

    if path.suffix.lower() == ".json":
        with open(path, "r") as f:
            raw = json.load(f)
        return {int(k): v for k, v in raw.items()}

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)

        # Flexible handling of common lookup column names
        code_col = find_field(df, ["code", "value", "class", "class_id", "raster_value"], required=False)
        label_col = find_field(df, ["label", "name", "class_name", "description"], required=False)

        if code_col is None or label_col is None:
            if df.shape[1] < 2:
                raise ValueError(f"Cannot infer lookup columns in {path}")
            code_col, label_col = df.columns[:2]

        return dict(zip(df[code_col].astype(int), df[label_col].astype(str)))

    raise ValueError(f"Unsupported lookup format: {path}")


def find_lookup_for_var(var_name, lookup_dir):
    lookup_dir = Path(lookup_dir)
    candidates = [
        lookup_dir / f"{var_name}_labels.json",
        lookup_dir / f"{var_name}_labels.csv",
        lookup_dir / f"{var_name}.json",
        lookup_dir / f"{var_name}.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    return None


def add_label_column(df, code_col, lookup, label_suffix="_label"):
    def lab(v):
        if pd.isna(v):
            return np.nan
        try:
            return lookup.get(int(v), np.nan)
        except Exception:
            return np.nan

    df[f"{code_col}{label_suffix}"] = df[code_col].apply(lab)
    return df


def load_mtbs_band_names(schema_csv, mtbs_src):
    schema = pd.read_csv(schema_csv)

    # Flexible schema column detection
    name_col = find_field(
        schema,
        ["band_name", "name", "variable", "var", "description"],
        required=False
    )

    band_col = find_field(
        schema,
        ["band", "band_index", "band_number", "index"],
        required=False
    )

    if name_col is None:
        raise ValueError(
            f"Could not infer MTBS variable-name column from schema. "
            f"Columns found: {list(schema.columns)}"
        )

    if band_col is not None:
        schema = schema.sort_values(band_col)

    names = [clean_name(x) for x in schema[name_col].tolist()]

    if len(names) != mtbs_src.count:
        raise ValueError(
            f"MTBS schema has {len(names)} names but raster has {mtbs_src.count} bands."
        )

    return names


def infer_join_key(left, right):
    candidates = [
        "PointID", "point_id", "POINT_ID",
        "PlotID", "plot_id", "PLOT_ID",
        "SiteID", "site_id", "SITE_ID",
        "SampleID", "sample_id", "SAMPLE_ID",
        "GlobalID", "globalid", "GLOBALID",
        "OBJECTID", "ObjectID", "FID"
    ]

    left_cols = {c.lower(): c for c in left.columns}
    right_cols = {c.lower(): c for c in right.columns}

    for cand in candidates:
        if cand.lower() in left_cols and cand.lower() in right_cols:
            return left_cols[cand.lower()], right_cols[cand.lower()]

    return None, None


# ============================================================
# LOAD POINTS AND FILTER TO 2025
# ============================================================

points = pd.read_csv(points_csv)

year_field = find_field(points, YEAR_FIELD_CANDIDATES)
if X_FIELD not in points.columns or Y_FIELD not in points.columns:
    raise ValueError(f"Expected coordinate fields '{X_FIELD}' and '{Y_FIELD}' in point file.")

points_2025 = points.loc[points[year_field] == YEAR_KEEP].copy()
points_2025 = points_2025.reset_index(drop=True)
points_2025["cube_row_id"] = np.arange(len(points_2025))

print(f"Loaded {len(points):,} total points.")
print(f"Retained {len(points_2025):,} points from {YEAR_KEEP}.")


# ============================================================
# JOIN VEGETATION TABLE
# Master points: PlotID, Year
# Vegetation:    Plot, Year
# ============================================================

veg = pd.read_csv(veg_csv)

required_point_cols = ["PlotID", "Year"]
required_veg_cols = ["Plot", "Year"]

missing_points = [c for c in required_point_cols if c not in points_2025.columns]
missing_veg = [c for c in required_veg_cols if c not in veg.columns]

if missing_points:
    raise ValueError(f"Missing point columns: {missing_points}")
if missing_veg:
    raise ValueError(f"Missing vegetation columns: {missing_veg}")

# Normalize join keys defensively
points_2025["PlotID_join"] = points_2025["PlotID"].astype(str).str.strip()
points_2025["Year_join"] = points_2025["Year"].astype(int)

veg["Plot_join"] = veg["Plot"].astype(str).str.strip()
veg["Year_join"] = veg["Year"].astype(int)

# Optional duplicate check
dup_veg = veg.duplicated(["Plot_join", "Year_join"], keep=False)
if dup_veg.any():
    dup_tbl = veg.loc[dup_veg, ["Plot", "Year"]].sort_values(["Year", "Plot"])
    dup_path = out_dir / "vegetation_duplicate_plot_year_keys.csv"
    dup_tbl.to_csv(dup_path, index=False)
    raise ValueError(
        f"Vegetation table has duplicate Plot-Year keys. "
        f"Wrote duplicates to: {dup_path}"
    )

cube = points_2025.merge(
    veg,
    left_on=["PlotID_join", "Year_join"],
    right_on=["Plot_join", "Year_join"],
    how="left",
    suffixes=("", "_veg")
)

n_unmatched = cube["Plot"].isna().sum()
print(f"Cube after vegetation join: {cube.shape[0]:,} rows, {cube.shape[1]:,} columns.")
print(f"Unmatched vegetation rows: {n_unmatched:,}")

if n_unmatched > 0:
    unmatched_path = out_dir / "unmatched_2025_point_vegetation_keys.csv"
    cube.loc[cube["Plot"].isna(), ["PlotID", "Year", "X", "Y"]].to_csv(unmatched_path, index=False)
    print(f"Wrote unmatched keys:\n{unmatched_path}")

# ============================================================
# SAMPLE POLARIS
# ============================================================

print("Sampling POLARIS...")
polaris_df = sample_multiband_raster(
    polaris_tif,
    cube,
    X_FIELD,
    Y_FIELD,
    prefix="polaris",
    band_names=POLARIS_BAND_NAMES
)
cube = pd.concat([cube, polaris_df], axis=1)


# ============================================================
# SAMPLE TOPO
# ============================================================

print("Sampling topography...")
topo_df = sample_multiband_raster(
    topo_tif,
    cube,
    X_FIELD,
    Y_FIELD,
    prefix="topo"
)
cube = pd.concat([cube, topo_df], axis=1)

# ============================================================
# SAMPLE SPECTRAL / HARMONIC FEATURES
# ============================================================

print("Sampling spectral/harmonic features...")

spectral_df = sample_multiband_raster(
    spectral_tif,
    cube,
    X_FIELD,
    Y_FIELD,
    prefix="s2"
)

cube = pd.concat([cube, spectral_df], axis=1)

# ============================================================
# SAMPLE LTDL RASTERS
# ============================================================

print("Sampling LTDL rasters...")

ltdl_tifs = sorted(list(ltdl_dir.glob("*.tif")) + list(ltdl_dir.glob("*.tiff")))

if len(ltdl_tifs) == 0:
    print(f"No LTDL rasters found in: {ltdl_dir}")
else:
    for tif in ltdl_tifs:
        var = clean_name(tif.stem)
        print(f"  {tif.name}")

        sampled = sample_multiband_raster(
            tif,
            cube,
            X_FIELD,
            Y_FIELD,
            prefix="ltdl"
        )

        # If single band, rename cleaner: ltdl_<rasterstem>
        if sampled.shape[1] == 1:
            sampled.columns = [f"ltdl_{var}"]

            lookup_path = find_lookup_for_var(var, ltdl_lookup_dir)
            if lookup_path is not None:
                lookup = read_lookup_table(lookup_path)
                sampled = add_label_column(sampled, f"ltdl_{var}", lookup)

        cube = pd.concat([cube, sampled], axis=1)


# ============================================================
# SAMPLE MTBS
# ============================================================

print("Sampling MTBS...")

with rasterio.open(mtbs_tif) as mtbs_src:
    mtbs_band_names = load_mtbs_band_names(mtbs_schema_csv, mtbs_src)

mtbs_df = sample_multiband_raster(
    mtbs_tif,
    cube,
    X_FIELD,
    Y_FIELD,
    prefix="mtbs",
    band_names=mtbs_band_names
)

# Remove duplicate prefix if schema already starts with mtbs_
mtbs_df.columns = [
    c.replace("mtbs_mtbs_", "mtbs_") for c in mtbs_df.columns
]

# Apply MTBS categorical labels
for var, lookup in MTBS_CATEGORICAL_LABELS.items():
    if var in mtbs_df.columns:
        mtbs_df = add_label_column(mtbs_df, var, lookup)
    else:
        print(f"Warning: MTBS categorical variable not found in sampled columns: {var}")

# Apply MTBS nodata handling for continuous/count variables
for var, meta in MTBS_CONTINUOUS_OR_COUNT_VARS.items():
    if var in mtbs_df.columns:
        nd = meta.get("nodata")
        if nd is not None:
            mtbs_df[var] = mtbs_df[var].replace(nd, np.nan)
    else:
        print(f"Warning: MTBS continuous/count variable not found in sampled columns: {var}")

cube = pd.concat([cube, mtbs_df], axis=1)


# ============================================================
# FINAL QA/QC
# ============================================================

# Drop exact duplicate columns if any were created by joins
cube = cube.loc[:, ~cube.columns.duplicated()].copy()

# Basic coordinate QA
bad_xy = cube[X_FIELD].isna() | cube[Y_FIELD].isna()
if bad_xy.any():
    print(f"Warning: {bad_xy.sum()} rows have missing X/Y coordinates.")

print(f"Final cube shape: {cube.shape[0]:,} rows x {cube.shape[1]:,} columns.")


# ============================================================
# WRITE OUTPUTS
# ============================================================

cube.to_csv(out_csv, index=False)
print(f"Wrote cube:\n{out_csv}")

metadata = {
    "year_subset": YEAR_KEEP,
    "points_csv": str(points_csv),
    "vegetation_csv": str(veg_csv),
    "ltdl_dir": str(ltdl_dir),
    "ltdl_lookup_dir": str(ltdl_lookup_dir),
    "polaris_tif": str(polaris_tif),
    "topo_tif": str(topo_tif),
    "mtbs_schema_csv": str(mtbs_schema_csv),
    "mtbs_tif": str(mtbs_tif),
    "n_points": int(cube.shape[0]),
    "n_columns": int(cube.shape[1]),
}

meta_path = out_dir / "NCA_2025_point_cube_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Wrote metadata:\n{meta_path}")

Loaded 618 total points.
Retained 118 points from 2025.
Cube after vegetation join: 118 rows, 94 columns.
Unmatched vegetation rows: 0
Sampling POLARIS...
Sampling topography...
Sampling spectral/harmonic features...
Sampling LTDL rasters...
  mgmt_binary_10m_new.tif
  mgmt_most_recent_time_since_10m_new.tif
  mgmt_most_recent_year_10m_new.tif
  mgmt_num_units_10m_new.tif
  mgmt_seeded_10m_new.tif
  mgmt_time_since_10m_new.tif
  mgmt_times_treated_10m_new.tif
  mgmt_treatment_type_10m_new.tif
  mgmt_trt_major_10m_new.tif
  mgmt_trt_sub_10m_new.tif
Sampling MTBS...
Final cube shape: 118 rows x 309 columns.
Wrote cube:
C:\NCA_DATA\Point_Cubes\NCA_2025_point_cube_noNLDAS.csv
Wrote metadata:
C:\NCA_DATA\Point_Cubes\NCA_2025_point_cube_metadata.json
